### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "SMOTE" / "gs_rf_SMOTE_midthres_2026-05-06" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.74,,,,,
1,0,cf_1,,,,,,,18.1551,,,0.9114,1,False,0.2426,0.2497
2,0,cf_2,,,,6,,,17.5596,,,0.9375,2,False,0.2426,0.497
3,0,cf_3,,,6,,,,16.7623,,,0.9724,2,False,0.2426,0.1698
4,0,cf_4,,,,,,,16.1324,5,,1.0,2,False,0.2426,0.1942
5,0,cf_5,,,,7,,,18.9745,7,,0.8755,3,False,0.2426,0.1617
6,0,cf_6,,,5,,,,18.9619,2,,0.8761,3,True,0.2426,0.1011
7,0,cf_7,3,,,,,,18.9619,3,,0.8761,3,False,0.2426,0.1673
8,0,cf_8,,,6,,,,18.9565,2,,0.8763,3,False,0.2426,0.1398
9,0,cf_9,2,,,,,,18.9039,3,,0.8786,3,False,0.2426,0.1552


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.74,,,,,
9,0,cf_6,,,5,,,,18.9619,2,,0.8761,3,True,0.2426,0.1011
1,1,xin,3,4,1,2,3,0,22.3757,0,2.99,,,,,
10,1,cf_4,,,2,,1,,22.3757,,,0.8796,3,True,0.4963,0.1956
11,1,cf_6,,,,3,1,,22.3757,,,0.8796,3,True,0.4963,0.1538
12,1,cf_8,,,,,1,,22.3757,1,,0.8796,3,True,0.4963,0.1855
13,1,cf_9,2,,,,1,,22.3757,,,0.8796,3,True,0.4963,0.1084
2,2,xin,5,3,1,1,4,0,22.694,7,3.14,,,,,
14,2,cf_10,,,2,6,1,,22.68,,,0.8759,4,True,0.3535,0.1313
3,3,xin,3,3,6,6,2,0,24.3418,1,106.62,,,,,


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.74,,,,,
9,0,cf_6,,,5,,,,18.9619,2,,0.8761,3,True,0.2426,0.1011
1,1,xin,3,4,1,2,3,0,22.3757,0,2.99,,,,,
10,1,cf_6,,,,3,1,,22.3757,,,0.8796,3,True,0.4963,0.1538
2,2,xin,5,3,1,1,4,0,22.694,7,3.14,,,,,
11,2,cf_10,,,2,6,1,,22.68,,,0.8759,4,True,0.3535,0.1313
3,3,xin,3,3,6,6,2,0,24.3418,1,106.62,,,,,
12,3,,,,,,,,,,,,,False,,
4,4,xin,5,4,2,7,1,0,21.2585,3,4.72,,,,,
13,4,,,,,,,,,,,,,False,,


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_6 ---
Predicted risk: 0.1011
Valid: True
Changes:
  cgtsmok: 3 → 5
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 2


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_4 ---
Predicted risk: 0.1956
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 → 1

--- cf_6 ---
Predicted risk: 0.1538
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_8 ---
Predicted risk: 0.1855
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_9 ---
Predicted risk: 0.1084
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original